# Mental Health Prediction on ELSA — Data Analysis & Harmonisation

## Project Overview

This notebook covers the **data preparation and exploratory analysis** phase of a machine-learning project that predicts mental health outcomes in the **English Longitudinal Study of Ageing (ELSA)**.

ELSA is a longitudinal cohort study of adults aged 50+ in England, with data collected across **10 survey waves** spanning 2002–2021. Each wave contains responses from thousands of participants on health, social, economic, and lifestyle factors.

### Prediction Targets
Three binary mental health outcomes were selected:
- **Depression** (`hepsyde`) — participant reported a psychiatric problem: depression
- **Anxiety** (`hepsyan`) — participant reported a psychiatric problem: anxiety
- **Emotional Problems** (`hepsyem`) — participant reported a psychiatric problem: emotional problems

### Key Challenges
1. **Cross-wave coding inconsistency** — Waves 1 and 2 encode psychiatric conditions differently from Waves 3–10, requiring custom derivation logic.
2. **Proprietary missing-value codes** — ELSA uses numeric sentinel codes (-1, -2, -8, -9) for non-response that must be converted to `NA`.
3. **Sparse positive classes** — Mental health conditions are relatively rare in this general population sample, creating class-imbalance challenges.

### Notebook Structure
1. Environment setup and data loading  
2. Target variable identification and harmonisation across all ten waves  
3. Exploratory class-balance analysis and target selection  
4. Candidate predictor definition based on clinical relevance  
5. Predictor harmonisation across all waves  
6. Final baseline predictor set selection  

The output of this notebook feeds into the companion modelling notebook.


## 1. Environment Setup

Google Colab is used as the compute environment. Mounting Google Drive gives access to the ELSA Stata (`.dta`) dataset files stored there.


In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## 2. Loading the ELSA Wave Data

Each of the ten ELSA waves is stored as a separate Stata file. All ten are loaded into individual Pandas DataFrames (`wave1` through `wave10`).

`convert_categoricals=False` is critical: it preserves the **raw numeric codes** for all variables rather than converting them to string labels. The harmonisation logic later in the notebook matches on specific numeric values (e.g. -1, -8, -9 for missing; 1 = yes, 2 = no), so the raw codes must be retained.


In [2]:
import pandas as pd
from google.colab import files
wave1 = pd.read_stata("/content/drive/MyDrive/ELSA Dataset/wave_1_core_data_v3.dta", convert_categoricals=False)
wave2 = pd.read_stata("/content/drive/MyDrive/ELSA Dataset/wave_2_core_data_v4.dta", convert_categoricals=False)
wave3 = pd.read_stata("/content/drive/MyDrive/ELSA Dataset/wave_3_elsa_data_v4.dta", convert_categoricals=False)
wave4 = pd.read_stata("/content/drive/MyDrive/ELSA Dataset/wave_4_elsa_data_v3.dta", convert_categoricals=False)
wave5 = pd.read_stata("/content/drive/MyDrive/ELSA Dataset/wave_5_elsa_data_v4.dta", convert_categoricals=False)
wave6 = pd.read_stata("/content/drive/MyDrive/ELSA Dataset/wave_6_elsa_data_v2.dta", convert_categoricals=False)
wave7 = pd.read_stata("/content/drive/MyDrive/ELSA Dataset/wave_7_elsa_data.dta", convert_categoricals=False)
wave8 = pd.read_stata("/content/drive/MyDrive/ELSA Dataset/wave_8_elsa_data_eul_v2.dta", convert_categoricals=False)
wave9 = pd.read_stata("/content/drive/MyDrive/ELSA Dataset/wave_9_elsa_data_eul_v1.dta", convert_categoricals=False)
wave10 = pd.read_stata("/content/drive/MyDrive/ELSA Dataset/wave_10_elsa_data_eul_v1.dta", convert_categoricals=False)



## 3. Target Variable Identification

### Psychiatric Outcome Variables in ELSA

In Waves 3–10, ELSA records self-reported psychiatric conditions as a set of separate binary flag variables:

| Variable | Condition |
|---|---|
| `hepsyde` | Depression |
| `hepsyan` | Anxiety |
| `hepsyem` | Emotional problems |
| `hepsyma` | Manic depression |
| `hepsysc` | Schizophrenia |
| `hepsyps` | Psychosis |
| `hepsyha` | Hallucinations |
| `hepsy95` | Other psychiatric problem |

### Harmonisation Challenge: Waves 1 and 2

Waves 1 and 2 store psychiatric conditions differently. A **gating question** first asks whether the participant has ever had a psychiatric problem (`heyrc` in Wave 1, `HeYrc` in Wave 2; 1 = yes, 2 = no). For those who answered yes, a series of **repeated-response columns** (`hepsy1`–`hepsy9` in Wave 1; `HePsy1`–`HePsy6` in Wave 2) records which specific conditions were mentioned using numeric codes:

| Code | Condition |
|---|---|
| 1 | Hallucinations → `hepsyha` |
| 2 | Anxiety → `hepsyan` |
| 3 | Depression → `hepsyde` |
| 4 | Emotional problems → `hepsyem` |
| 5 | Schizophrenia → `hepsysc` |
| 6 | Psychosis → `hepsyps` |
| 8 | Manic depression → `hepsyma` |
| 95 | Other → `hepsy95` |

The eight binary target flags must therefore be **derived** for Waves 1 and 2 by parsing these repeated-response columns. They cannot be read directly as in later waves.

Additionally, Wave 10 names one target `HEPSY95` (uppercase) rather than the canonical `hepsy95`, requiring a simple column rename.


### Checking Which Target Columns Already Exist in Waves 1 and 2

Before deriving the targets, we verify which canonical target column names are present in Waves 1 and 2, and list all columns starting with `hepsy` or `HePsy`. This diagnostic step confirms that the standard binary flags are absent and guides the derivation approach.


In [3]:
import pandas as pd

# columns you want to harmonise to
target_cols = [
    "hepsyha", "hepsyan", "hepsyde", "hepsyem",
    "hepsysc", "hepsyps", "hepsyma", "hepsy95"
]

# check whether these columns already exist in wave1 and wave2
existing_wave1 = [col for col in target_cols if col in wave1.columns]
existing_wave2 = [col for col in target_cols if col in wave2.columns]

print("Existing target columns in wave1:", existing_wave1)
print("Existing target columns in wave2:", existing_wave2)

# optional: show all columns starting with hepsy / HePsy
print("\nWave1 columns starting with 'hepsy':")
print([col for col in wave1.columns if col.lower().startswith("hepsy")])

print("\nWave2 columns starting with 'hepsy':")
print([col for col in wave2.columns if col.lower().startswith("hepsy")])

Existing target columns in wave1: []
Existing target columns in wave2: []

Wave1 columns starting with 'hepsy':
['hepsy1', 'hepsy2', 'hepsy3', 'hepsy4', 'hepsy5', 'hepsy6', 'hepsy7', 'hepsy8', 'hepsy9']

Wave2 columns starting with 'hepsy':
['HePsy1', 'HePsy2', 'HePsy3', 'HePsy4', 'HePsy5', 'HePsy6', 'HePsya', 'HePsyb', 'HePsyc', 'HePsyd', 'HePsye']


## 4. Harmonising Target Variables — Waves 1 and 2

Two custom functions derive the eight binary target columns for Waves 1 and 2 from the repeated-response structure.

**`derive_wave1_targets` (Wave 1):**
- If `heyrc` = 2 (no psychiatric problem ever), all eight targets are set to **0**.
- If `heyrc` = 1 (yes), each repeated slot (`hepsy1`–`hepsy9`) is checked. If any slot contains code 3, `hepsyde` is set to **1**; code 2 → `hepsyan` = 1; and so on.
- Rows where `heyrc` is missing or ambiguous are left as **NA** across all targets.
- Sentinel codes -1, -8, -9 in the repeated slots are treated as `NA` before parsing.

**`derive_wave2_targets` (Wave 2):**
- Identical logic, using `HeYrc` as the gating question and `HePsy1`–`HePsy6` as the repeated columns.

This approach correctly handles three distinct respondent states: *confirmed no problem* (0), *confirmed specific problem* (1), and *status unknown* (NA).


In [4]:
import pandas as pd

target_cols = ["hepsyha","hepsyan","hepsyde","hepsyem","hepsysc","hepsyps","hepsyma","hepsy95"]

target_map = {
    1: "hepsyha",
    2: "hepsyan",
    3: "hepsyde",
    4: "hepsyem",
    5: "hepsysc",
    6: "hepsyps",
    8: "hepsyma",
    95: "hepsy95"
}

def derive_wave1_targets(df):
    df = df.copy()
    psy_cols = [f"hepsy{i}" for i in range(1, 10) if f"hepsy{i}" in df.columns]

    # clean source codes
    temp = df[psy_cols].replace([-1, -8, -9], pd.NA)

    # initialize new columns
    for _, new_col in target_map.items():
        df[new_col] = pd.NA

    # base question: 1=yes, 2=no
    yes_mask = df["heyrc"] == 1
    no_mask = df["heyrc"] == 2

    # if respondent said NO psychiatric problem, set all targets to 0
    for _, new_col in target_map.items():
        df.loc[no_mask, new_col] = 0

    # if respondent said YES, mark specific conditions from repeated hepsy fields
    for code, new_col in target_map.items():
        mentioned = temp.eq(code).any(axis=1)
        df.loc[yes_mask, new_col] = mentioned[yes_mask].astype("Int64")

    return df

def derive_wave2_targets(df):
    df = df.copy()
    psy_cols = [f"HePsy{i}" for i in range(1, 7) if f"HePsy{i}" in df.columns]

    temp = df[psy_cols].replace([-1, -8, -9], pd.NA)

    for _, new_col in target_map.items():
        df[new_col] = pd.NA

    yes_mask = df["HeYrc"] == 1
    no_mask = df["HeYrc"] == 2

    for _, new_col in target_map.items():
        df.loc[no_mask, new_col] = 0

    for code, new_col in target_map.items():
        mentioned = temp.eq(code).any(axis=1)
        df.loc[yes_mask, new_col] = mentioned[yes_mask].astype("Int64")

    return df

# IMPORTANT: assign back
wave1 = derive_wave1_targets(wave1)
wave2 = derive_wave2_targets(wave2)

# Wave 10 harmonisation
if "HEPSY95" in wave10.columns and "hepsy95" not in wave10.columns:
    wave10["hepsy95"] = wave10["HEPSY95"]

### Sanity Check: Target Counts After Wave 1 and 2 Harmonisation

The derived target columns are inspected for both waves. For each target we check: which columns were successfully created, the count of positive cases (1), negative cases (0), and missing values (NA). This is the first quality gate, confirming that the derivation logic produced plausible class distributions.


In [5]:
target_cols = ["hepsyha","hepsyan","hepsyde","hepsyem","hepsysc","hepsyps","hepsyma","hepsy95"]

print("Wave1 target columns created:", [c for c in target_cols if c in wave1.columns])
print("Wave2 target columns created:", [c for c in target_cols if c in wave2.columns])
for col in target_cols:
    print(f"\nWave1 {col}")
    print(wave1[col].value_counts(dropna=False))

for col in target_cols:
    print(f"\nWave2 {col}")
    print(wave2[col].value_counts(dropna=False))

Wave1 target columns created: ['hepsyha', 'hepsyan', 'hepsyde', 'hepsyem', 'hepsysc', 'hepsyps', 'hepsyma', 'hepsy95']
Wave2 target columns created: ['hepsyha', 'hepsyan', 'hepsyde', 'hepsyem', 'hepsysc', 'hepsyps', 'hepsyma', 'hepsy95']

Wave1 hepsyha
hepsyha
<NA>    11192
0         895
1          12
Name: count, dtype: int64

Wave1 hepsyan
hepsyan
<NA>    11192
0         536
1         371
Name: count, dtype: int64

Wave1 hepsyde
hepsyde
<NA>    11192
0         457
1         450
Name: count, dtype: int64

Wave1 hepsyem
hepsyem
<NA>    11192
0         749
1         158
Name: count, dtype: int64

Wave1 hepsysc
hepsysc
<NA>    11192
0         896
1          11
Name: count, dtype: int64

Wave1 hepsyps
hepsyps
<NA>    11192
0         902
1           5
Name: count, dtype: int64

Wave1 hepsyma
hepsyma
<NA>    11192
0         878
1          29
Name: count, dtype: int64

Wave1 hepsy95
hepsy95
<NA>    11192
0         871
1          36
Name: count, dtype: int64

Wave2 hepsyha
hepsyha
<NA>    874

## 5. Harmonising Target Variables — Waves 3 to 10

For Waves 3–10, the psychiatric targets exist as direct binary columns but still require cleaning:

1. **Replace missing-value sentinel codes**: -1 (not applicable), -2 (not asked), -3, -4, -8 (don't know), and -9 (refused) are all replaced with `pd.NA`.
2. **Rename Wave 10 column**: `HEPSY95` → `hepsy95` for consistency.
3. **Cast to `Int64`**: Pandas nullable integer type, which supports `NA` without converting the column to float.

After cleaning all waves, they are stored in a dictionary `waves = {1: wave1, 2: wave2, ..., 10: wave10}` for convenient iteration in subsequent cells.


In [6]:
import pandas as pd

target_cols = ["hepsyha","hepsyan","hepsyde","hepsyem","hepsysc","hepsyps","hepsyma","hepsy95"]
missing_codes = [-1, -2, -8, -9]

def clean_later_wave_targets(df):
    df = df.copy()

    if "HEPSY95" in df.columns and "hepsy95" not in df.columns:
        df["hepsy95"] = df["HEPSY95"]

    for col in target_cols:
        if col in df.columns:
            df[col] = df[col].replace(missing_codes, pd.NA).astype("Int64")

    return df

wave3 = clean_later_wave_targets(wave3)
wave4 = clean_later_wave_targets(wave4)
wave5 = clean_later_wave_targets(wave5)
wave6 = clean_later_wave_targets(wave6)
wave7 = clean_later_wave_targets(wave7)
wave8 = clean_later_wave_targets(wave8)
wave9 = clean_later_wave_targets(wave9)
wave10 = clean_later_wave_targets(wave10)

waves = {
    1: wave1, 2: wave2, 3: wave3, 4: wave4, 5: wave5,
    6: wave6, 7: wave7, 8: wave8, 9: wave9, 10: wave10
}

### Cross-Wave Target Column Coverage Check

Each wave is inspected to confirm which of the eight target columns are present and which are absent. Missing columns at this stage indicate harmonisation gaps. This completeness check ensures all waves have consistent structure before the waves are combined.


In [7]:
for w, df in waves.items():
    present = [c for c in target_cols if c in df.columns]
    missing = [c for c in target_cols if c not in df.columns]
    print(f"Wave {w}")
    print("Present:", present)
    print("Missing:", missing)
    print("-" * 40)

Wave 1
Present: ['hepsyha', 'hepsyan', 'hepsyde', 'hepsyem', 'hepsysc', 'hepsyps', 'hepsyma', 'hepsy95']
Missing: []
----------------------------------------
Wave 2
Present: ['hepsyha', 'hepsyan', 'hepsyde', 'hepsyem', 'hepsysc', 'hepsyps', 'hepsyma', 'hepsy95']
Missing: []
----------------------------------------
Wave 3
Present: ['hepsyha', 'hepsyan', 'hepsyde', 'hepsyem', 'hepsysc', 'hepsyps', 'hepsyma', 'hepsy95']
Missing: []
----------------------------------------
Wave 4
Present: ['hepsyha', 'hepsyan', 'hepsyde', 'hepsyem', 'hepsysc', 'hepsyps', 'hepsyma', 'hepsy95']
Missing: []
----------------------------------------
Wave 5
Present: ['hepsyha', 'hepsyan', 'hepsyde', 'hepsyem', 'hepsysc', 'hepsyps', 'hepsyma', 'hepsy95']
Missing: []
----------------------------------------
Wave 6
Present: ['hepsyha', 'hepsyan', 'hepsyde', 'hepsyem', 'hepsysc', 'hepsyps', 'hepsyma', 'hepsy95']
Missing: []
----------------------------------------
Wave 7
Present: ['hepsyha', 'hepsyan', 'hepsyde', 'h

### Full Value-Count Inspection Across All Waves

Detailed value counts for every target in every wave are printed to:
- Confirm no sentinel codes (-1, -8, -9) remain after cleaning
- Inspect per-wave class balance (ratio of positive to negative cases)
- Identify targets that are nearly entirely missing in some waves, making them unsuitable as modelling targets


In [8]:
for w, df in waves.items():
    print(f"\n========== Wave {w} ==========")
    for col in target_cols:
        print(f"\n{col}")
        print(df[col].value_counts(dropna=False))


========== Wave 1 ==========

hepsyha
hepsyha
<NA>    11192
0         895
1          12
Name: count, dtype: int64

hepsyan
hepsyan
<NA>    11192
0         536
1         371
Name: count, dtype: int64

hepsyde
hepsyde
<NA>    11192
0         457
1         450
Name: count, dtype: int64

hepsyem
hepsyem
<NA>    11192
0         749
1         158
Name: count, dtype: int64

hepsysc
hepsysc
<NA>    11192
0         896
1          11
Name: count, dtype: int64

hepsyps
hepsyps
<NA>    11192
0         902
1           5
Name: count, dtype: int64

hepsyma
hepsyma
<NA>    11192
0         878
1          29
Name: count, dtype: int64

hepsy95
hepsy95
<NA>    11192
0         871
1          36
Name: count, dtype: int64

========== Wave 2 ==========

hepsyha
hepsyha
<NA>    8747
0        685
Name: count, dtype: int64

hepsyan
hepsyan
<NA>    8747
0        682
1          3
Name: count, dtype: int64

hepsyde
hepsyde
<NA>    8747
0        683
1          2
Name: count, dtype: int64

hepsyem
hepsyem
<NA>    87

### Summary Table: Positive-Case Prevalence by Wave and Target

A compact summary table replaces the lengthy per-wave value-count output. For each wave–target combination, the table shows:
- **Non-missing percentage** — proportion of respondents with a valid (non-NA) response for that target
- **Positive-case percentage among non-missing** — within-wave prevalence of the psychiatric condition

This table is the key diagnostic for selecting which targets have sufficient case numbers and wave coverage to support modelling.


In [9]:
target_cols = ["hepsyha","hepsyan","hepsyde","hepsyem","hepsysc","hepsyps","hepsyma","hepsy95"]

summary = []

for w, df in waves.items():
    n = len(df)
    row = {"wave": w, "n_rows": n}
    for col in target_cols:
        non_missing = df[col].notna().sum()
        positive = (df[col] == 1).sum()
        negative = (df[col] == 0).sum()

        row[f"{col}_nonmissing_pct"] = round(non_missing / n * 100, 2)
        row[f"{col}_positive_pct_among_nonmissing"] = round(
            positive / non_missing * 100, 2
        ) if non_missing > 0 else pd.NA
    summary.append(row)

summary_df = pd.DataFrame(summary)
summary_df

,wave,n_rows,hepsyha_nonmissing_pct,hepsyha_positive_pct_among_nonmissing,hepsyan_nonmissing_pct,hepsyan_positive_pct_among_nonmissing,hepsyde_nonmissing_pct,hepsyde_positive_pct_among_nonmissing,hepsyem_nonmissing_pct,hepsyem_positive_pct_among_nonmissing,hepsysc_nonmissing_pct,hepsysc_positive_pct_among_nonmissing,hepsyps_nonmissing_pct,hepsyps_positive_pct_among_nonmissing,hepsyma_nonmissing_pct,hepsyma_positive_pct_among_nonmissing,hepsy95_nonmissing_pct,hepsy95_positive_pct_among_nonmissing
0,1,12099,7.50,1.32,7.50,40.90,7.50,49.61,7.50,17.42,7.50,1.21,7.50,0.55,7.50,3.20,7.50,3.97
1,2,9432,7.26,0.00,7.26,0.44,7.26,0.29,7.26,0.15,7.26,0.00,7.26,0.00,7.26,0.00,7.26,0.00
2,3,9771,7.84,1.83,7.84,54.70,7.84,73.11,7.84,22.85,7.84,1.83,7.84,1.31,7.84,3.52,7.84,8.22
3,4,11050,8.63,1.99,8.64,56.02,8.64,71.31,8.64,25.03,8.64,1.47,8.64,1.05,8.64,4.19,8.64,8.38
4,5,10274,9.63,1.42,9.63,55.21,9.63,69.26,9.63,20.12,9.63,1.11,9.63,1.21,9.63,3.03,9.63,7.28
5,6,10601,10.35,2.46,10.35,53.51,10.35,66.27,10.35,20.97,10.35,1.19,10.35,2.28,10.35,3.28,10.35,8.20
6,7,9666,10.46,2.37,10.46,51.53,10.46,68.45,10.46,19.49,10.46,1.68,10.46,1.58,10.46,3.07,10.46,8.01
7,8,8445,1.37,6.03,1.37,55.17,1.37,59.48,1.37,20.69,1.37,0.86,1.37,1.72,1.37,3.45,1.37,13.79
8,9,8736,3.12,3.30,3.12,63.00,3.12,69.23,3.12,18.32,3.12,2.56,3.12,2.20,3.12,3.30,3.12,5.13
9,10,7586,15.19,1.65,15.19,61.63,15.19,68.92,15.19,19.62,15.19,1.04,15.19,1.65,15.19,2.86,15.19,7.64


## 6. Exploratory Analysis: Combined Outcome (`any_psych_problem`)

Inspection of the summary table shows that several psychiatric subtypes — hallucinations, schizophrenia, psychosis — have very few positive cases per wave (sometimes fewer than 50 respondents). These are too sparse to support reliable binary classification models.

As a diagnostic step, all eight subtype flags are collapsed into a single binary outcome `any_psych_problem`:
- **1** if the respondent reported any of the eight conditions
- **0** if they explicitly reported none
- **NA** if all eight fields are missing

This gives the maximum achievable positive-case count and shows whether a single combined target would be viable.


In [10]:
for w, df in waves.items():
    temp = df[target_cols].copy()

    all_missing = temp.isna().all(axis=1)
    any_psych = temp.fillna(0).max(axis=1).astype("Int64")
    any_psych[all_missing] = pd.NA

    waves[w]["any_psych_problem"] = any_psych

for w, df in waves.items():
     print(f"\nWave {w}")
     print(df["any_psych_problem"].value_counts(dropna=False))


Wave 1
any_psych_problem
<NA>    11192
1         572
0         335
Name: count, dtype: Int64

Wave 2
any_psych_problem
<NA>    8747
0        682
1          3
Name: count, dtype: Int64

Wave 3
any_psych_problem
<NA>    9005
1        759
0          7
Name: count, dtype: Int64

Wave 4
any_psych_problem
<NA>    10095
1         946
0           9
Name: count, dtype: Int64

Wave 5
any_psych_problem
<NA>    9285
1        978
0         11
Name: count, dtype: Int64

Wave 6
any_psych_problem
<NA>    9504
1       1088
0          9
Name: count, dtype: Int64

Wave 7
any_psych_problem
<NA>    8655
1        999
0         12
Name: count, dtype: Int64

Wave 8
any_psych_problem
<NA>    8329
1        114
0          2
Name: count, dtype: Int64

Wave 9
any_psych_problem
<NA>    8463
1        272
0          1
Name: count, dtype: Int64

Wave 10
any_psych_problem
<NA>    6434
1       1145
0          7
Name: count, dtype: Int64


/tmp/ipykernel_1502/2840906699.py:5: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  any_psych = temp.fillna(0).max(axis=1).astype("Int64")
/tmp/ipykernel_1502/2840906699.py:5: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  any_psych = temp.fillna(0).max(axis=1).astype("Int64")


### Why the Combined Outcome Was Rejected for Modelling

The `any_psych_problem` counts reveal a problem in the opposite direction: combining all eight subtypes produces a positive-case proportion so large that negative cases (respondents with no psychiatric condition) become the minority class in many waves. A workable classifier needs sufficient examples of **both** classes.

**Decision**: Select the three subtypes with the highest and most stable prevalence as individual modelling targets — these are depression, anxiety, and emotional problems. These three offer:
- Enough positive cases (hundreds per wave) to train a reliable model
- Enough negative controls to prevent trivially biased predictions
- Consistent non-missing coverage across most of the ten waves


**Selected Main Target Variables:**
Depression (hepsyde)
and Anxiety (hepsyan)

**Selected Secondary Target Variable:** Emotional Problems (hepsyem)



This is the **final target rebuild** for the three selected main outcomes: depression (`hepsyde`), anxiety (`hepsyan`), and emotional problems (`hepsyem`). For Waves 1 and 2, the targets are rebuilt from repeated `hepsy`/`HePsy` mentions only. For Waves 3 to 10, the direct flags are cleaned and standardised. This cell harmonises target variables in all waves.

## 7. Final Target Variable Rebuild (Three Outcomes Only)

The target variables are rebuilt from scratch, keeping only the three selected outcomes: depression (`hepsyde`), anxiety (`hepsyan`), and emotional problems (`hepsyem`).

**`rebuild_from_repeated_mentions_small` (Waves 1–2)**:
- For each target, scans the repeated-response columns for the matching condition code.
- A row is coded **0** if at least one usable response slot is observed but none contains the target code.
- A row is coded **1** if any slot contains the target code.
- A row is left as **NA** if all slots are missing or flagged as don't-know / refused.
- This correctly distinguishes *confirmed absence* (0) from *unknown status* (NA).

**`rebuild_from_direct_flags_small` (Waves 3–10)**:
- Reads the existing binary flags directly.
- Codes -1 and -2 (not applicable / not asked) → **0**, because these indicate the question was not relevant rather than that the answer is unknown.
- Codes -3, -4, -8, -9 → **NA**, because these represent genuine non-response.
- All values outside {0, 1} are forced to NA.

This rebuild produces a clean, consistent `{0, 1, NA}` encoding for all three targets across all ten waves, superseding the earlier harmonisation attempts.


In [18]:
import pandas as pd

# only the three targets you want
target_map_small = {
    2: "hepsyan",   # anxiety
    3: "hepsyde",   # depression
    4: "hepsyem"    # emotional problems
}

target_cols_small = ["hepsyde", "hepsyan", "hepsyem"]

def rebuild_from_repeated_mentions_small(df, source_cols):
    """
    For Waves 1-2:
    Build depression/anxiety/emotional-problems from repeated HePsy/hepsy columns.
    """
    df = df.copy()
    raw = df[source_cols].apply(pd.to_numeric, errors="coerce")

    # row has usable info if at least one repeated slot is not refusal/dk/missing
    observed_any = (~raw.isin([-8, -9])) & raw.notna()
    observed_any = observed_any.any(axis=1)

    for code, out_col in target_map_small.items():
        out = pd.Series(pd.NA, index=df.index, dtype="Int64")
        out.loc[observed_any] = 0
        out.loc[raw.eq(code).any(axis=1)] = 1
        df[out_col] = out

    return df

def rebuild_from_direct_flags_small(df):
    """
    For Waves 3-10:
    Keep direct 0/1 flags for the three targets.
    Treat -1/-2 as 0, and -3/-4/-8/-9 as NA.
    """
    df = df.copy()

    for col in target_cols_small:
        if col not in df.columns:
            continue

        s = pd.to_numeric(df[col], errors="coerce")
        s = s.replace({
            -1: 0,
            -2: 0,
            -3: pd.NA,
            -4: pd.NA,
            -8: pd.NA,
            -9: pd.NA
        })

        s = s.where(s.isin([0, 1]), pd.NA).astype("Int64")
        df[col] = s

    return df

# Waves 1-2
w1_psy_cols = [f"hepsy{i}" for i in range(1, 10) if f"hepsy{i}" in wave1.columns]
w2_psy_cols = [f"HePsy{i}" for i in range(1, 7) if f"HePsy{i}" in wave2.columns]

wave1 = rebuild_from_repeated_mentions_small(wave1, w1_psy_cols)
wave2 = rebuild_from_repeated_mentions_small(wave2, w2_psy_cols)

# Waves 3-10
wave3 = rebuild_from_direct_flags_small(wave3)
wave4 = rebuild_from_direct_flags_small(wave4)
wave5 = rebuild_from_direct_flags_small(wave5)
wave6 = rebuild_from_direct_flags_small(wave6)
wave7 = rebuild_from_direct_flags_small(wave7)
wave8 = rebuild_from_direct_flags_small(wave8)
wave9 = rebuild_from_direct_flags_small(wave9)
wave10 = rebuild_from_direct_flags_small(wave10)

### Confirming Final Target Counts After Rebuild

The `waves` dictionary is refreshed and the final value counts for depression, anxiety, and emotional problems are printed for every wave. This is the definitive quality check confirming that the rebuilt targets have sensible class distributions before the waves are merged into a modelling dataset.


In [12]:
waves = {
    1: wave1, 2: wave2, 3: wave3, 4: wave4, 5: wave5,
    6: wave6, 7: wave7, 8: wave8, 9: wave9, 10: wave10
}

for w, df in waves.items():
    print(f"\n========== Wave {w} ==========")
    for col in ["hepsyde", "hepsyan", "hepsyem"]:
        print(f"\n{col}")
        print(df[col].value_counts(dropna=False))


========== Wave 1 ==========

hepsyde
hepsyde
0    11431
1      668
Name: count, dtype: Int64

hepsyan
hepsyan
0    11577
1      522
Name: count, dtype: Int64

hepsyem
hepsyem
0    11880
1      219
Name: count, dtype: Int64

========== Wave 2 ==========

hepsyde
hepsyde
0    9267
1     165
Name: count, dtype: Int64

hepsyan
hepsyan
0    9289
1     143
Name: count, dtype: Int64

hepsyem
hepsyem
0    9371
1      61
Name: count, dtype: Int64

========== Wave 3 ==========

hepsyde
hepsyde
<NA>    9005
1        560
0        206
Name: count, dtype: Int64

hepsyan
hepsyan
<NA>    9005
1        419
0        347
Name: count, dtype: Int64

hepsyem
hepsyem
<NA>    9005
0        591
1        175
Name: count, dtype: Int64

========== Wave 4 ==========

hepsyde
hepsyde
<NA>    10095
1         681
0         274
Name: count, dtype: Int64

hepsyan
hepsyan
<NA>    10095
1         535
0         420
Name: count, dtype: Int64

hepsyem
hepsyem
<NA>    10095
0         716
1         239
Name: count, dtype: I

## 8. Building the Combined Longitudinal Dataset

### Stacking All Waves Into a Single DataFrame

All ten harmonised waves are concatenated vertically into `combined_targets`, a long-format DataFrame where each row is one participant observation in one wave. The unique participant identifier (`idauniq`) and wave number are retained.

> **Note**: This DataFrame contains only the three target variables and identifiers — no predictor features yet. It is used solely for class-balance verification across the full longitudinal sample.


## Important note about this section

The next two cells (`combined_targets` and the first `dep_df`/`anx_df`/`emo_df`) are **target-only exploratory cells**.

They are useful for checking class balance, but they are **not** the modelling dataset.

Later in the notebook, **Cell 45** rebuilds the real modelling dataset from `harmonised_waves`.  
When checking train/validation/test sizes, use the `dep_df`, `anx_df`, and `emo_df` created in **Cell 45**, not the earlier ones created here.


In [13]:
import pandas as pd

waves = {
    1: wave1, 2: wave2, 3: wave3, 4: wave4, 5: wave5,
    6: wave6, 7: wave7, 8: wave8, 9: wave9, 10: wave10
}

target_cols = ["hepsyde", "hepsyan", "hepsyem"]

combined_list = []

for w, df in waves.items():
    temp = df[["idauniq"] + target_cols].copy()
    temp["wave"] = w
    combined_list.append(temp)

combined_targets = pd.concat(combined_list, ignore_index=True)

print(combined_targets.shape)
print(combined_targets.head())

(97660, 5)
   idauniq  hepsyde  hepsyan  hepsyem  wave
0   100035        0        0        0     1
1   100036        0        0        0     1
2   100037        0        0        0     1
3   100038        0        0        0     1
4   100039        0        0        0     1


### Creating Target-Specific Datasets and Verifying Class Balance

Three separate datasets are created — one per outcome — by dropping rows where that target's value is missing. The class counts are printed for each dataset, confirming the total positive and negative case counts available across all waves for each modelling task. These numbers determine whether the dataset is large enough to support a reliable classification model.


In [14]:
dep_df = combined_targets.dropna(subset=["hepsyde"]).copy()
anx_df = combined_targets.dropna(subset=["hepsyan"]).copy()
emo_df = combined_targets.dropna(subset=["hepsyem"]).copy()

print("Depression target counts")
print(dep_df["hepsyde"].value_counts(dropna=False))

print("\nAnxiety target counts")
print(anx_df["hepsyan"].value_counts(dropna=False))

print("\nEmotional problems target counts")
print(emo_df["hepsyem"].value_counts(dropna=False))

Depression target counts
hepsyde
0    22660
1     5230
Name: count, dtype: Int64

Anxiety target counts
hepsyan
0    23671
1     4219
Name: count, dtype: Int64

Emotional problems target counts
hepsyem
0    26270
1     1620
Name: count, dtype: Int64


## 9. Predictor Selection Strategy

### Why Data-Driven Variable Importance Screening Was Not Applied to All Raw Variables

ELSA contains hundreds of raw variables per wave, but directly computing importance scores across the full raw variable pool would be methodologically flawed for this dataset. The key reasons are:

1. **Wave-specific and non-harmonisable variables**: Many columns exist in only one or two waves. Including them without harmonisation produces biased importance estimates that do not generalise.
2. **Target leakage risk**: ELSA's survey routing means some variables are only presented to respondents who have already reported a psychiatric condition. Using such variables as predictors would give the model access to information derived from the target — a form of data leakage.
3. **Duplicate constructs**: Multiple versions of the same concept (e.g. different occupational classification schemes) appear across waves. An unconstrained search would assign high importance to all versions without providing independent signal.
4. **Administrative and identifier columns**: Interview dates, sampling weights, and interviewer IDs carry no substantive predictive information.

### Chosen Approach: Theory-Driven Candidate Pool

Candidate predictors were selected based on three criteria:
1. **Clinical and epidemiological relevance** — variables with established or plausible associations with depression, anxiety, or emotional problems in older adults
2. **Harmonisability across waves** — variables that can be consistently identified and coded across most or all ten waves
3. **No leakage risk** — variables measured independently of the psychiatric outcome, not as follow-up questions conditional on a positive response

This approach — defining a valid candidate pool first, then applying empirical selection within it — is standard practice in epidemiological prediction modelling. It avoids the pitfalls of unconstrained importance screening while still allowing the final predictor set to be informed by data.


### Candidate Predictor Concepts

The candidate pool covers sociodemographic, health, and lifestyle factors that are well-established correlates of depression, anxiety, and emotional problems in older adults.

**First-pass candidates** (harmonised and included in the initial model pipeline):
- Age, sex, relationship status, paid employment status
- Self-rated health, long-standing illness, limiting illness
- Occupational class (NS-SEC), smoking status, physical activity
- Pain, household size, children outside household, wealth / assets

**Second-pass candidates** (added progressively after the baseline pipeline was validated):
- Functional limitation scores (IADL / ADL difficulty)
- Cognitive score
- Alcohol use frequency
- Pain severity

This phased approach allowed the harmonisation and modelling pipeline to be validated on a manageable predictor set before expanding.


In [15]:
candidate_predictors = [
    "age",
    "sex",
    "relationship_status",
    "paid_employment",
    "self_rated_health",
    "long_standing_illness",
    "limiting_illness",
    "occupational_class_nssec",
    "smoking_status",
    "physical_activity",
    "pain",
    "household_size",
    "child_outside_household",
    "wealth_or_assets"
]

second_pass_predictors = [
    "iadl_difficulty",
    "adl_difficulty",
    "cognitive_score",
    "alcohol_frequency",
    "pain_severity",
    "detailed_cognitive_measures"
]

## 10. Re-Running the Target Rebuild Before Predictor Harmonisation

The target variables are rebuilt once more at this point as a deliberate safeguard. If cells were re-run out of order earlier in the notebook, the `wave1`–`wave10` DataFrames could be in a stale or inconsistent state. Running the rebuild here guarantees that all waves contain clean, up-to-date `{0, 1, NA}` psychiatric flags before any predictor columns are attached in the following steps.


In [19]:
import pandas as pd

# only the three targets you want
target_map_small = {
    2: "hepsyan",   # anxiety
    3: "hepsyde",   # depression
    4: "hepsyem"    # emotional problems
}

target_cols_small = ["hepsyde", "hepsyan", "hepsyem"]

def rebuild_from_repeated_mentions_small(df, source_cols):
    """
    For Waves 1-2:
    Build depression/anxiety/emotional-problems from repeated HePsy/hepsy columns.
    """
    df = df.copy()
    raw = df[source_cols].apply(pd.to_numeric, errors="coerce")

    # row has usable info if at least one repeated slot is not refusal/dk/missing
    observed_any = (~raw.isin([-8, -9])) & raw.notna()
    observed_any = observed_any.any(axis=1)

    for code, out_col in target_map_small.items():
        out = pd.Series(pd.NA, index=df.index, dtype="Int64")
        out.loc[observed_any] = 0
        out.loc[raw.eq(code).any(axis=1)] = 1
        df[out_col] = out

    return df

def rebuild_from_direct_flags_small(df):
    """
    For Waves 3-10:
    Keep direct 0/1 flags for the three targets.
    Treat -1/-2 as 0, and -3/-4/-8/-9 as NA.
    """
    df = df.copy()

    for col in target_cols_small:
        if col not in df.columns:
            continue

        s = pd.to_numeric(df[col], errors="coerce")
        s = s.replace({
            -1: 0,
            -2: 0,
            -3: pd.NA,
            -4: pd.NA,
            -8: pd.NA,
            -9: pd.NA
        })

        s = s.where(s.isin([0, 1]), pd.NA).astype("Int64")
        df[col] = s

    return df

# Waves 1-2
w1_psy_cols = [f"hepsy{i}" for i in range(1, 10) if f"hepsy{i}" in wave1.columns]
w2_psy_cols = [f"HePsy{i}" for i in range(1, 7) if f"HePsy{i}" in wave2.columns]

wave1 = rebuild_from_repeated_mentions_small(wave1, w1_psy_cols)
wave2 = rebuild_from_repeated_mentions_small(wave2, w2_psy_cols)

# Waves 3-10
wave3 = rebuild_from_direct_flags_small(wave3)
wave4 = rebuild_from_direct_flags_small(wave4)
wave5 = rebuild_from_direct_flags_small(wave5)
wave6 = rebuild_from_direct_flags_small(wave6)
wave7 = rebuild_from_direct_flags_small(wave7)
wave8 = rebuild_from_direct_flags_small(wave8)
wave9 = rebuild_from_direct_flags_small(wave9)
wave10 = rebuild_from_direct_flags_small(wave10)

### Refreshing `waves` Dictionary and Confirming Target Counts

The `waves` dictionary is updated to reference the freshly rebuilt wave DataFrames. The final value counts for all three targets are printed once more as a confirmation that the rebuild ran correctly and the targets are in the expected `{0, 1, NA}` format before predictor harmonisation begins.


In [20]:
waves = {
    1: wave1, 2: wave2, 3: wave3, 4: wave4, 5: wave5,
    6: wave6, 7: wave7, 8: wave8, 9: wave9, 10: wave10
}

for w, df in waves.items():
    print(f"\n========== Wave {w} ==========")
    for col in ["hepsyde", "hepsyan", "hepsyem"]:
        print(f"\n{col}")
        print(df[col].value_counts(dropna=False))


========== Wave 1 ==========

hepsyde
hepsyde
0    11431
1      668
Name: count, dtype: Int64

hepsyan
hepsyan
0    11577
1      522
Name: count, dtype: Int64

hepsyem
hepsyem
0    11880
1      219
Name: count, dtype: Int64

========== Wave 2 ==========

hepsyde
hepsyde
0    9267
1     165
Name: count, dtype: Int64

hepsyan
hepsyan
0    9289
1     143
Name: count, dtype: Int64

hepsyem
hepsyem
0    9371
1      61
Name: count, dtype: Int64

========== Wave 3 ==========

hepsyde
hepsyde
<NA>    9005
1        560
0        206
Name: count, dtype: Int64

hepsyan
hepsyan
<NA>    9005
1        419
0        347
Name: count, dtype: Int64

hepsyem
hepsyem
<NA>    9005
0        591
1        175
Name: count, dtype: Int64

========== Wave 4 ==========

hepsyde
hepsyde
<NA>    10095
1         681
0         274
Name: count, dtype: Int64

hepsyan
hepsyan
<NA>    10095
1         535
0         420
Name: count, dtype: Int64

hepsyem
hepsyem
<NA>    10095
0         716
1         239
Name: count, dtype: I

## 11. Predictor Harmonisation Across All Waves

The candidate predictors must be harmonised before the waves can be stacked into a single modelling dataset. Harmonisation involves:

1. **Source column identification**: Variable names differ across waves (e.g. age may be `indager` in some waves and `Indager` in others). A *selector function* for each concept returns the correct column name for a given DataFrame.
2. **Response coding standardisation**: All binary variables are mapped to a consistent 1 = yes / 0 = no scale; ordinal and numeric variables retain their natural scale with a standard sign convention.
3. **Missing-value replacement**: All ELSA sentinel codes (-1, -2, -3, -4, -8, -9) are replaced with `pd.NA`.
4. **Dtype casting**: Binary variables → `Int64` (nullable integer); continuous variables → `Float64` (nullable float).

Harmonised columns are named with an `_h` suffix (e.g. `age_h`, `sex_h`) to clearly distinguish them from the raw source columns.

A **source audit table** is generated at the end of this cell, recording which raw column was used as the source for each predictor in each wave. This table is used to identify predictor concepts that could not be harmonised in certain waves (i.e. where the source column was `None`).


## Important note before this cell

This cell rebuilds `harmonised_waves` from the current `waves` object.

If the target counts later look stale, the most common cause is that this cell was run **before** the final target rebuild cell.  
The safe run order is:

1. Final target rebuild (**Cell 25**)  
2. Rebuild `waves` (**Cell 27**)  
3. Rebuild `harmonised_waves` (**this cell**)  
4. Add extra predictors (**Cell 40**)  
5. Build the modelling dataset (**Cell 45**)


In [21]:
import pandas as pd
import numpy as np
from IPython.display import display

# -----------------------------
# helpers
# -----------------------------

COMMON_MISSING = {-1, -2, -3, -4, -8, -9}

def first_existing(df, candidates):
    for c in candidates:
        if c in df.columns:
            return c
    return None

def clean_numeric(s):
    return pd.to_numeric(s, errors="coerce")

def clean_special_missing(s, extra_missing=None):
    s = clean_numeric(s)
    miss = set(COMMON_MISSING)
    if extra_missing:
        miss |= set(extra_missing)
    return s.replace(list(miss), pd.NA)

def value_counts_safe(df, col):
    if col not in df.columns:
        return f"{col} not found"
    return df[col].value_counts(dropna=False).sort_index()

# -----------------------------
# source selectors by wave
# -----------------------------

def pick_age_var(df):
    # prefer definitive age where present
    return first_existing(df, ["indager", "Indager", "dhager"])

def pick_sex_var(df):
    # prefer definitive sex where present
    return first_existing(df, ["indsex", "Indsex", "dhsex", "DhSex"])

def pick_relationship_var(df):
    # prefer derived relationship variables; raw dhr/DhR only as fallback
    return first_existing(df, ["couple", "futype", "dhr", "DhR"])

def pick_srh_var(df):
    return first_existing(df, ["hehelf", "Hehelf"])

def pick_long_ill_var(df):
    return first_existing(df, ["heill", "Heill"])

def pick_limiting_ill_var(df):
    return first_existing(df, ["helim", "Helim"])

def pick_work_limit_var(df):
    return first_existing(df, ["helwk", "HeLWk"])

def pick_paid_work_var(df):
    # prefer clearer current work indicator first
    return first_existing(df, ["iawork", "dhwork", "DhWork", "astwork", "aeconact", "anactiv"])

def pick_nssec_var(df, wave_num):
    # prefer 3-cat; then 5-cat; then 8-cat; then Wave 1 long version
    candidates = [
        f"w{wave_num}nssec3", f"w{wave_num}nssec5", f"w{wave_num}nssec8",
        "w2nssec3", "w2nssec5", "w2nssec8",
        "w3nssec3", "w3nssec5", "w3nssec8",
        "w4nssec3", "w4nssec5", "w4nssec8",
        "w5nssec3", "w5nssec5", "w5nssec8",
        "w6nssec3", "w6nssec5", "w6nssec8",
        "w7nssec3", "w7nssec5", "w7nssec8",
        "w8nssec3", "w8nssec5", "w8nssec8",
        "w9nssec3", "w9nssec5", "w9nssec8",
        "w10nssec3", "w10nssec5", "w10nssec8",
        "anssec"
    ]
    return first_existing(df, candidates)

# -----------------------------
# harmonisers
# -----------------------------

def harmonise_age(df):
    src = pick_age_var(df)
    out = pd.Series(pd.NA, index=df.index, dtype="Float64")
    if src is None:
        return out, src

    s = clean_numeric(df[src])

    # disclosure-protected top-coded age values
    s = s.replace({99: 90, -7: 90})
    s = s.where(s >= 0, pd.NA)

    return s.astype("Float64"), src

def harmonise_sex(df):
    """
    0 = male
    1 = female
    """
    src = pick_sex_var(df)
    out = pd.Series(pd.NA, index=df.index, dtype="Int64")
    if src is None:
        return out, src

    s = clean_special_missing(df[src])
    out.loc[s == 1] = 0
    out.loc[s == 2] = 1
    return out, src

def harmonise_relationship(df):
    """
    1 = partnered
    0 = not partnered
    """
    src = pick_relationship_var(df)
    out = pd.Series(pd.NA, index=df.index, dtype="Int64")
    if src is None:
        return out, src

    s = clean_special_missing(df[src])

    if src == "couple":
        out.loc[s.isin([1, 2])] = 1   # married/cohabit
        out.loc[s == 3] = 0           # neither

    elif src == "futype":
        out.loc[s.isin([2, 3])] = 1   # couple separate/joint finances
        out.loc[s == 1] = 0           # single

    elif src in ["dhr", "DhR"]:
        out.loc[s.isin([1, 2])] = 1   # husband/wife or partner/cohabitee
        out.loc[s.notna() & ~s.isin([1, 2])] = 0

    return out, src

def harmonise_srh(df):
    """
    1 = excellent ... 5 = poor
    """
    src = pick_srh_var(df)
    out = pd.Series(pd.NA, index=df.index, dtype="Int64")
    if src is None:
        return out, src

    s = clean_special_missing(df[src])
    out = s.where(s.isin([1, 2, 3, 4, 5]), pd.NA).astype("Int64")
    return out, src

def harmonise_yes_no(df, picker):
    """
    For variables coded 1 = yes, 2 = no
    Returns 1=yes, 0=no
    """
    src = picker(df)
    out = pd.Series(pd.NA, index=df.index, dtype="Int64")
    if src is None:
        return out, src

    s = clean_special_missing(df[src])
    out.loc[s == 1] = 1
    out.loc[s == 2] = 0
    return out, src

def harmonise_paid_employment(df):
    """
    1 = in paid employment
    0 = not in paid employment
    """
    src = pick_paid_work_var(df)
    out = pd.Series(pd.NA, index=df.index, dtype="Int64")
    if src is None:
        return out, src

    s = clean_special_missing(df[src])

    if src in ["iawork", "dhwork", "DhWork", "astwork"]:
        out.loc[s == 1] = 1
        out.loc[s == 2] = 0

    elif src == "aeconact":
        # Wave 1 HSE economic status:
        # 1 = In employment, others not in paid employment
        out.loc[s == 1] = 1
        out.loc[s.isin([2, 3, 4])] = 0

    elif src == "anactiv":
        # Wave 1 HSE activity last week:
        # 2 = in paid employment or self-employed
        out.loc[s == 2] = 1
        out.loc[s.isin([1, 3, 4, 5, 6, 7, 8, 9, 10, 11])] = 0

    return out, src

def harmonise_nssec3(df, wave_num):
    """
    Output:
      1 = managerial/professional
      2 = intermediate
      3 = routine/manual
    Sets 'other' or unclassifiable codes to NA.
    """
    src = pick_nssec_var(df, wave_num)
    out = pd.Series(pd.NA, index=df.index, dtype="Int64")
    if src is None:
        return out, src

    s = clean_numeric(df[src])

    # common special codes across waves
    s = s.replace([-1, -2, -3, -4, -6, -8, -9, 99], pd.NA)

    if src.endswith("nssec3"):
        out.loc[s == 1] = 1
        out.loc[s == 2] = 2
        out.loc[s == 3] = 3

    elif src.endswith("nssec5"):
        # pragmatic mapping from 5-cat to 3-cat
        out.loc[s == 1] = 1
        out.loc[s.isin([2, 3])] = 2
        out.loc[s.isin([4, 5])] = 3

    elif src.endswith("nssec8"):
        # pragmatic mapping from 8-cat to 3-cat
        out.loc[s.isin([1, 2])] = 1
        out.loc[s.isin([3, 4])] = 2
        out.loc[s.isin([5, 6, 7])] = 3

    elif src == "anssec":
        # pragmatic Wave 1 fallback from long version to broad 3 classes
        main = pd.to_numeric(s, errors="coerce")
        main_floor = np.floor(main)

        out.loc[main_floor.isin([1, 2, 3, 4, 5, 6])] = 1
        out.loc[main_floor.isin([7, 8, 9])] = 2
        out.loc[main_floor.isin([10, 11, 12, 13, 14])] = 3
        # 15,16,17 left as NA

    return out, src

# -----------------------------
# apply harmonisation
# -----------------------------

harmonised_waves = {}
source_audit = []

for w, df in waves.items():
    df = df.copy()

    df["age_h"], age_src = harmonise_age(df)
    df["sex_h"], sex_src = harmonise_sex(df)
    df["relationship_status_h"], rel_src = harmonise_relationship(df)
    df["self_rated_health_h"], srh_src = harmonise_srh(df)
    df["long_standing_illness_h"], ill_src = harmonise_yes_no(df, pick_long_ill_var)
    df["limiting_illness_h"], lim_src = harmonise_yes_no(df, pick_limiting_ill_var)

    # Wave 1: no clear health-limits-work variable found
    if w == 1:
        df["health_limits_work_h"] = pd.Series(pd.NA, index=df.index, dtype="Int64")
        wk_lim_src = None
    else:
        df["health_limits_work_h"], wk_lim_src = harmonise_yes_no(df, pick_work_limit_var)

    df["paid_employment_h"], pay_src = harmonise_paid_employment(df)
    df["occupational_class_nssec_h"], nssec_src = harmonise_nssec3(df, w)

    harmonised_waves[w] = df

    source_audit.append({
        "wave": w,
        "age_src": age_src,
        "sex_src": sex_src,
        "relationship_src": rel_src,
        "self_rated_health_src": srh_src,
        "long_illness_src": ill_src,
        "limiting_illness_src": lim_src,
        "health_limits_work_src": wk_lim_src,
        "paid_employment_src": pay_src,
        "nssec_src": nssec_src
    })

source_audit_df = pd.DataFrame(source_audit)
display(source_audit_df)

,wave,age_src,sex_src,relationship_src,self_rated_health_src,long_illness_src,limiting_illness_src,health_limits_work_src,paid_employment_src,nssec_src
0,1,indager,indsex,futype,hehelf,heill,helim,None,iawork,anssec
1,2,indager,indsex,couple,Hehelf,Heill,Helim,HeLWk,iawork,w2nssec3
2,3,indager,indsex,couple,None,heill,helim,helwk,iawork,w3nssec3
3,4,indager,indsex,couple,hehelf,heill,helim,helwk,iawork,w4nssec3
4,5,indager,indsex,couple,hehelf,heill,helim,helwk,iawork,w5nssec3
5,6,indager,indsex,couple,Hehelf,Heill,Helim,HeLWk,DhWork,w6nssec3
6,7,indager,indsex,couple,Hehelf,Heill,Helim,HeLWk,DhWork,None
7,8,indager,indsex,couple,hehelf,heill,helim,helwk,iawork,w8nssec3
8,9,indager,indsex,couple,hehelf,heill,helim,helwk,iawork,w9nssec3
9,10,indager,indsex,couple,Hehelf,Heill,Helim,HeLWk,DhWork,None


### Source Audit Results: Identifying Unavailable Predictors

Inspection of the source audit table reveals three predictors that cannot be harmonised in some waves:
- **Occupational class (NS-SEC)** is unavailable in Waves 6 and 10
- **Self-rated health** source column is absent in Wave 3
- **Health limits work** has no suitable source column in Wave 1

Predictors missing in multiple waves introduce substantial imputation burden and reduce cross-wave consistency. These three variables are therefore **dropped from the first-pass predictor set** and the analysis continues with the remaining harmonisable variables. Self-rated health and limiting illness are revisited at a later stage (see the modelling notebook).


## 12. Extending the Harmonised Dataset: Pain, Household Size, and Children Outside Household

Three additional predictor concepts are harmonised and appended to `harmonised_waves`:

- **`pain_h`**: Whether the respondent is often troubled by pain (1 = yes, 0 = no). Pain has a well-documented bidirectional relationship with depression and anxiety in older adults.
- **`household_size_h`**: Total number of people in the household (continuous). Captures aspects of social support and living situation.
- **`child_outside_household_h`**: Whether the respondent has children living outside the household (1 = yes, 0 = no). A proxy for social network and potential support.

After harmonisation, a combined modelling DataFrame is built and missing-value rates are checked for each predictor across the full longitudinal sample.


## Important note before this cell

This cell adds `pain_h`, `child_outside_household_h`, and `household_size_h` to `harmonised_waves`.

It must be run **after** the main predictor harmonisation cell (**Cell 37**) and **before** the modelling dataset is rebuilt in **Cell 45**.


In [22]:
import pandas as pd
from IPython.display import display

# ---------------------------------
# selector functions for new vars
# ---------------------------------

def pick_pain_var(df):
    return first_existing(df, ["hepain", "HePain"])

def pick_child_outside_var(df):
    # prefer simple yes/no indicator; fallback to total number outside household
    return first_existing(df, ["chouthh", "ChOutHh", "chotot", "ChoTot"])

def pick_household_size_var(df):
    return first_existing(df, ["hhtot", "HHTot", "ahhsize", "Ahhsize"])


# ---------------------------------
# harmonisers for new vars
# ---------------------------------

def harmonise_pain(df):
    """
    pain_h:
      1 = often troubled with pain / pain present
      0 = no
    Assumes 1=yes, 2=no where available.
    """
    src = pick_pain_var(df)
    out = pd.Series(pd.NA, index=df.index, dtype="Int64")
    if src is None:
        return out, src

    s = clean_special_missing(df[src])

    # handle common yes/no codings
    out.loc[s == 1] = 1
    out.loc[s.isin([0, 2])] = 0

    return out, src


def harmonise_child_outside_household(df):
    """
    child_outside_household_h:
      1 = yes
      0 = no

    If source is:
      - chouthh: usually 1=yes, 2=no
      - chotot: >0 yes, 0 no
    """
    src = pick_child_outside_var(df)
    out = pd.Series(pd.NA, index=df.index, dtype="Int64")
    if src is None:
        return out, src

    s = clean_special_missing(df[src])

    if src.lower() == "chouthh":
        out.loc[s == 1] = 1
        out.loc[s == 2] = 0
    elif src.lower() == "chotot":
        out.loc[s > 0] = 1
        out.loc[s == 0] = 0

    return out, src


def harmonise_household_size(df):
    """
    household_size_h:
      numeric count of people in household
    """
    src = pick_household_size_var(df)
    out = pd.Series(pd.NA, index=df.index, dtype="Float64")
    if src is None:
        return out, src

    s = clean_numeric(df[src])

    # treat standard special codes as missing
    s = s.replace([-1, -2, -3, -4, -8, -9], pd.NA)

    # household size must be positive
    s = s.where(s > 0, pd.NA)

    return s.astype("Float64"), src


# ---------------------------------
# apply new harmonisation
# ---------------------------------

updated_waves = {}
extra_source_audit = []

for w, df in harmonised_waves.items():
    df = df.copy()

    df["pain_h"], pain_src = harmonise_pain(df)
    df["child_outside_household_h"], child_src = harmonise_child_outside_household(df)
    df["household_size_h"], hhsize_src = harmonise_household_size(df)

    updated_waves[w] = df

    extra_source_audit.append({
        "wave": w,
        "pain_src": pain_src,
        "child_outside_household_src": child_src,
        "household_size_src": hhsize_src
    })

harmonised_waves = updated_waves

extra_source_audit_df = pd.DataFrame(extra_source_audit)
display(extra_source_audit_df)

# merge into your existing audit table if it already exists
if "source_audit_df" in globals():
    source_audit_df = source_audit_df.merge(extra_source_audit_df, on="wave", how="left")
    display(source_audit_df)


# ---------------------------------
# quick check of counts by wave
# ---------------------------------

new_predictors = ["pain_h", "child_outside_household_h", "household_size_h"]

for w, df in harmonised_waves.items():
    print(f"\n========== Wave {w} ==========")
    for col in new_predictors:
        print(f"\n{col}")
        print(value_counts_safe(df, col))


# ---------------------------------
# build first-pass modelling dataset
# dropping the predictors you decided to leave out for now:
#   self_rated_health_h, health_limits_work_h, occupational_class_nssec_h
# ---------------------------------

model_predictors = [
    "age_h",
    "sex_h",
    "relationship_status_h",
    "long_standing_illness_h",
    "limiting_illness_h",
    "paid_employment_h",
    "pain_h",
    "child_outside_household_h",
    "household_size_h"
]

model_rows = []

for w, df in harmonised_waves.items():
    keep_cols = ["idauniq"] + model_predictors + ["hepsyde", "hepsyan", "hepsyem"]
    temp = df[keep_cols].copy()
    temp["wave"] = w
    model_rows.append(temp)

combined_model_df = pd.concat(model_rows, ignore_index=True)

print("combined_model_df shape:", combined_model_df.shape)
display(combined_model_df.head())


# ---------------------------------
# target-specific modelling datasets
# ---------------------------------

dep_model_df = combined_model_df.dropna(subset=["hepsyde"]).copy()
anx_model_df = combined_model_df.dropna(subset=["hepsyan"]).copy()
emo_model_df = combined_model_df.dropna(subset=["hepsyem"]).copy()

print("\nDepression target counts")
print(dep_model_df["hepsyde"].value_counts(dropna=False))

print("\nAnxiety target counts")
print(anx_model_df["hepsyan"].value_counts(dropna=False))

print("\nEmotional problems target counts")
print(emo_model_df["hepsyem"].value_counts(dropna=False))


# ---------------------------------
# optional: check missingness in predictors
# ---------------------------------

missing_summary = []

for col in model_predictors:
    missing_summary.append({
        "predictor": col,
        "missing_pct_in_combined_model_df": combined_model_df[col].isna().mean()
    })

missing_summary_df = pd.DataFrame(missing_summary).sort_values("missing_pct_in_combined_model_df", ascending=False)
display(missing_summary_df)

,wave,pain_src,child_outside_household_src,household_size_src
0,1,hepain,chouthh,hhtot
1,2,HePain,chouthh,HHTot
2,3,hepain,chouthh,hhtot
3,4,hepain,chouthh,hhtot
4,5,hepain,chouthh,hhtot
5,6,HePain,chouthh,HHTot
6,7,HePain,chouthh,HHTot
7,8,hepain,chouthh,hhtot
8,9,hepain,chouthh,hhtot
9,10,HePain,chouthh,HHTot


,wave,age_src,sex_src,relationship_src,self_rated_health_src,long_illness_src,limiting_illness_src,health_limits_work_src,paid_employment_src,nssec_src,pain_src,child_outside_household_src,household_size_src
0,1,indager,indsex,futype,hehelf,heill,helim,None,iawork,anssec,hepain,chouthh,hhtot
1,2,indager,indsex,couple,Hehelf,Heill,Helim,HeLWk,iawork,w2nssec3,HePain,chouthh,HHTot
2,3,indager,indsex,couple,None,heill,helim,helwk,iawork,w3nssec3,hepain,chouthh,hhtot
3,4,indager,indsex,couple,hehelf,heill,helim,helwk,iawork,w4nssec3,hepain,chouthh,hhtot
4,5,indager,indsex,couple,hehelf,heill,helim,helwk,iawork,w5nssec3,hepain,chouthh,hhtot
5,6,indager,indsex,couple,Hehelf,Heill,Helim,HeLWk,DhWork,w6nssec3,HePain,chouthh,HHTot
6,7,indager,indsex,couple,Hehelf,Heill,Helim,HeLWk,DhWork,None,HePain,chouthh,HHTot
7,8,indager,indsex,couple,hehelf,heill,helim,helwk,iawork,w8nssec3,hepain,chouthh,hhtot
8,9,indager,indsex,couple,hehelf,heill,helim,helwk,iawork,w9nssec3,hepain,chouthh,hhtot
9,10,indager,indsex,couple,Hehelf,Heill,Helim,HeLWk,DhWork,None,HePain,chouthh,HHTot



========== Wave 1 ==========

pain_h
pain_h
0       7386
1       4520
<NA>     193
Name: count, dtype: Int64

child_outside_household_h
child_outside_household_h
0    2746
1    9353
Name: count, dtype: Int64

household_size_h
household_size_h
1.0     2850
2.0     6664
3.0     1552
4.0      764
5.0      183
6.0       68
7.0        8
8.0        4
9.0        2
10.0       3
11.0       1
Name: count, dtype: Int64

========== Wave 2 ==========

pain_h
pain_h
0       5786
1       3513
<NA>     133
Name: count, dtype: Int64

child_outside_household_h
child_outside_household_h
0    1744
1    7688
Name: count, dtype: Int64

household_size_h
household_size_h
1.0     2338
2.0     5361
3.0     1127
4.0      424
5.0      127
6.0       44
7.0        7
8.0        2
10.0       1
11.0       1
Name: count, dtype: Int64

========== Wave 3 ==========

pain_h
pain_h
0       5942
1       3593
<NA>     236
Name: count, dtype: Int64

child_outside_household_h
child_outside_household_h
0    2325
1    7446
Name

,idauniq,age_h,sex_h,relationship_status_h,long_standing_illness_h,limiting_illness_h,paid_employment_h,pain_h,child_outside_household_h,household_size_h,hepsyde,hepsyan,hepsyem,wave
0,100035,67.0,1,1,1,0,0,1,1,2.0,0,0,0,1
1,100036,53.0,1,1,1,1,1,0,0,2.0,0,0,0,1
2,100037,68.0,1,1,0,<NA>,0,0,0,2.0,0,0,0,1
3,100038,64.0,1,1,0,<NA>,0,0,1,3.0,0,0,0,1
4,100039,53.0,0,1,1,1,1,1,1,3.0,0,0,0,1



Depression target counts
hepsyde
0    22660
1     5230
Name: count, dtype: Int64

Anxiety target counts
hepsyan
0    23671
1     4219
Name: count, dtype: Int64

Emotional problems target counts
hepsyem
0    26270
1     1620
Name: count, dtype: Int64


,predictor,missing_pct_in_combined_model_df
4,limiting_illness_h,0.450358
6,pain_h,0.042843
7,child_outside_household_h,0.022148
5,paid_employment_h,0.007383
8,household_size_h,0.002212
3,long_standing_illness_h,0.000717
0,age_h,0.000010
1,sex_h,0.000000
2,relationship_status_h,0.000000


### Dropping Predictors with Excessive Missingness

Inspection of the missing-value summary reveals two problematic predictors:
- **Limiting illness** has a very high missing percentage across waves, making imputation less reliable and its inclusion harder to justify.
- **Child outside household** shows no zero values (only 1s and NAs) in Wave 10, indicating the variable was not measured consistently in that wave.

Both variables are dropped from the baseline predictor set at this stage. They may be revisited in sensitivity analyses or if additional data cleaning resolves the coverage gaps.


## 13. Final Baseline Predictor Set

After the harmonisation process and coverage checks, seven predictors are retained for the first-pass modelling dataset. These are the variables with the best combination of clinical relevance, cross-wave availability, and low missingness.

The final baseline set is defined in the cell below and is passed directly to the modelling notebook as the starting predictor configuration.


In [23]:
baseline_predictors = [
    "age_h",
    "sex_h",
    "relationship_status_h",
    "long_standing_illness_h",
    "paid_employment_h",
    "pain_h",
    "household_size_h"
]